# ST-GCN - input stream comparison: joint vs bone vs joint+bone

A genuine single-model comparison, not an ensemble or a training trick - same architecture, same winning recipe (StepLR, top1-selection, advanced augmentation, weight_decay=1e-3, the strongest isolated single-model result from today's ablation), only the **input stream** changes between runs.

Motivated by mmaction2's own published metafile for this exact architecture on NTU60: bone input alone (91.69% top-1) meaningfully outperformed joint input alone (88.95%) on the same model. Worth testing whether that holds here too.

`compute_bone()` in our own `dataset.py` returns `(T, 17, 2)` - dx/dy only, no confidence channel (verified directly via grep before writing this, not assumed). Three variants:
- **joint**: the existing baseline, `in_channels=3` (x, y, confidence)
- **bone**: `in_channels=2` (dx, dy only)
- **joint_bone**: both concatenated, `in_channels=5`

Every final metric is logged to ClearML via `report_single_value` (the correct API for headline numbers, not the per-epoch `report_scalar` curve used for training progress), plus a confusion matrix and per-class precision/recall/F1 table - not just training curves this time.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
    confusion_matrix, precision_recall_fscore_support,
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### paths

In [2]:
OUT_ROOT = "/workspace/outputs/stgcn_bone_stream"
CKPT_ROOT = "/workspace/checkpoints/stgcn_bone_stream"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - advanced augmentation, matching the winning recipe

In [3]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="advanced")
val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=42)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### batch -> ST-GCN input, parameterized by stream

Uses only the two dict keys confirmed to exist by the grep before writing this: `batch["keypoints"]` (T,17,3) and `batch["bone"]` (T,17,2).

In [4]:
STREAM_CHANNELS = {"joint": 3, "bone": 2, "joint_bone": 5}


def batch_to_stgcn_input(batch, stream):
    if stream == "joint":
        x = batch["keypoints"].float()
    elif stream == "bone":
        x = batch["bone"].float()
    elif stream == "joint_bone":
        x = torch.cat([batch["keypoints"].float(), batch["bone"].float()], dim=-1)
    else:
        raise ValueError(f"unknown stream: {stream}")
    return x.unsqueeze(1).unsqueeze(1)   # (N, num_clips=1, M=1, T, V, C)


### SMOKE TEST - verify all three stream shapes before training anything

Checking the actual tensor shapes and channel counts match `STREAM_CHANNELS` before committing to a full run - cheap insurance given this is the first time `batch["bone"]` has been used in any notebook this project.

In [5]:
smoke_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))

print("raw batch[\"keypoints\"] shape:", smoke_batch["keypoints"].shape)
print("raw batch[\"bone\"] shape:", smoke_batch["bone"].shape)
print()

for stream, expected_c in STREAM_CHANNELS.items():
    x = batch_to_stgcn_input(smoke_batch, stream)
    actual_c = x.shape[-1]
    status = "OK" if actual_c == expected_c else "MISMATCH"
    print(f"{stream:12s} shape={tuple(x.shape)}  expected_channels={expected_c}  actual={actual_c}  [{status}]")
    assert actual_c == expected_c, f"{stream}: channel mismatch"

print()
print("all three stream shapes verified correct")


raw batch["keypoints"] shape: torch.Size([4, 30, 17, 3])
raw batch["bone"] shape: torch.Size([4, 30, 17, 2])

joint        shape=(4, 1, 1, 30, 17, 3)  expected_channels=3  actual=3  [OK]
bone         shape=(4, 1, 1, 30, 17, 2)  expected_channels=2  actual=2  [OK]
joint_bone   shape=(4, 1, 1, 30, 17, 5)  expected_channels=5  actual=5  [OK]

all three stream shapes verified correct


### metrics, including per-class breakdown for the ClearML table/confusion matrix

In [6]:
@torch.no_grad()
def evaluate(model, loader, stream):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_stgcn_input(batch, stream).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "labels": labels, "preds": preds,
    }


### the reusable training function

Same StepLR schedule, same weight_decay=1e-3, same top1-based checkpoint selection throughout - only `stream` (and therefore `in_channels`) varies between calls. Full ClearML reporting: per-epoch curve via report_scalar (as before), but now also every final test metric via report_single_value, a confusion matrix, and a per-class table.

In [7]:
def train_one_stream(stream, seed=42, num_epochs=60, patience=20, batch_size=16,
                      lr=0.01, weight_decay=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)

    in_channels = STREAM_CHANNELS[stream]

    model_cfg = dict(
        type="RecognizerGCN",
        backbone=dict(
            type="STGCN",
            graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
            in_channels=in_channels, base_channels=64, ch_ratio=2,
            num_stages=9, inflate_stages=[4, 7], down_stages=[4, 7],
        ),
        cls_head=dict(type="GCNHead", num_classes=num_classes, in_channels=256,
                      loss_cls=dict(type="CrossEntropyLoss")),
    )
    model = MODELS.build(model_cfg).to(device)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=balanced_sampler,
                               num_workers=4, pin_memory=True, persistent_workers=True)

    task = Task.init(project_name="BadmintonPoseAnalysis", task_name=f"stgcn_stream_{stream}", reuse_last_task_id=False)
    logger = task.get_logger()
    task.connect({
        "stream": stream, "in_channels": in_channels, "seed": seed,
        "lr": lr, "weight_decay": weight_decay, "num_epochs": num_epochs,
        "batch_size": batch_size, "schedule": "StepLR", "augmentation_policy": "advanced",
    })

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

    best_val_top1 = 0.0
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_labels, epoch_preds = [], []

        for batch in train_loader:
            x = batch_to_stgcn_input(batch, stream).to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(x, mode="tensor", stage="head")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_labels.append(y.cpu().numpy())
            epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

        scheduler.step()

        train_loss = epoch_loss / len(train_loader.dataset)
        train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))
        val_metrics = evaluate(model, val_loader, stream)

        logger.report_scalar("loss", "train", train_loss, iteration=epoch)
        logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
        logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
        logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"[{stream}] epoch {epoch:3d}  train_acc={train_acc:.4f}  "
                  f"val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

        if epochs_without_improvement >= patience:
            print(f"[{stream}] early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f}")
            break

    model.load_state_dict(best_state)

    train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True), stream)
    test_final = evaluate(model, test_loader, stream)

    metrics = {
        "stream": stream,
        "train_acc": train_final["top1"],
        "test_top1": test_final["top1"],
        "test_top5": test_final["top5"],
        "test_mca": test_final["mca"],
        "test_macro_f1": test_final["macro_f1"],
        "train_test_gap": train_final["top1"] - test_final["top1"],
    }

    # --- full ClearML reporting of final results, not just the training curve ---
    for metric_name in ["train_acc", "test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap"]:
        logger.report_single_value(name=metric_name, value=metrics[metric_name])

    cm = confusion_matrix(test_final["labels"], test_final["preds"], labels=list(range(num_classes)))
    logger.report_confusion_matrix(
        title="Test Confusion Matrix", series=stream,
        matrix=cm, xaxis="Predicted", yaxis="Actual",
        xlabels=class_names, ylabels=class_names,
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        test_final["labels"], test_final["preds"], labels=list(range(num_classes)), zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    logger.report_table(title="Per-Class Test Metrics", series=stream, table_plot=per_class_df)

    ckpt_path = os.path.join(CKPT_ROOT, f"stream_{stream}_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    task.upload_artifact("checkpoint", ckpt_path)
    per_class_df.to_csv(os.path.join(OUT_ROOT, f"per_class_{stream}.csv"), index=False)
    task.close()

    return metrics, per_class_df


### run all three streams

3x a single run in total - genuinely fine given no rush on the timeline, and each run is now properly documented in ClearML, not just a training curve.

In [8]:
streams_to_test = ["joint", "bone", "joint_bone"]

all_results = {}
all_per_class = {}

for stream in streams_to_test:
    print(f"\n=== training stream={stream} ===")
    metrics, per_class_df = train_one_stream(stream)
    all_results[stream] = metrics
    all_per_class[stream] = per_class_df
    print(f"[{stream}] FINAL: test_top1={metrics['test_top1']:.4f}  gap={metrics['train_test_gap']:.4f}  "
          f"test_mca={metrics['test_mca']:.4f}  test_macro_f1={metrics['test_macro_f1']:.4f}")



=== training stream=joint ===
ClearML Task: created new task id=82777e5101234a9e8ebc4fe06e0184fc
2026-09-10 07:14:42,423 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/82777e5101234a9e8ebc4fe06e0184fc/output/log
[joint] epoch   0  train_acc=0.2910  val_top1=0.5416  val_mca=0.4004
[joint] epoch  10  train_acc=0.8029  val_top1=0.6825  val_mca=0.6322
[joint] epoch  20  train_acc=0.9305  val_top1=0.7183  val_mca=0.6504
[joint] epoch  30  train_acc=0.9583  val_top1=0.7234  val_mca=0.6355
[joint] epoch  40  train_acc=0.9573  val_top1=0.7273  val_mca=0.6484
[joint] epoch  50  train_acc=0.9589  val_top1=0.7273  val_mca=0.6398
[joint] early stopping at epoch 57, best val_top1=0.7311


███████████████████████████████ 100% | 11.66/11.66 MB [00:04<00:00,  2.81MB/s]: 


[joint] FINAL: test_top1=0.7008  gap=0.2776  test_mca=0.5938  test_macro_f1=0.6159

=== training stream=bone ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=959b92af532a404d870ad7e183f67f60
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/959b92af532a404d870ad7e183f67f60/output/log
[bone] epoch   0  train_acc=0.2556  val_top1=0.3355  val_mca=0.2707
[bone] epoch  10  train_acc=0.7164  val_top1=0.6492  val_mca=0.5927
[bone] epoch  20  train_acc=0.8547  val_top1=0.7042  val_mca=0.6344
[bone] epoch  30  train_acc=0.8858  val_top1=0.7093  val_mca=0.6349
[bone] epoch  40  train_acc=0.8899  val_top1=0.7132  val_mca=0.6194
[bone] early stopping at epoch 42, best val_top1=0.7157


...

[bone] FINAL: test_top1=0.6829  gap=0.1917  test_mca=0.5660  test_macro_f1=0.5793

=== training stream=joint_bone ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=25583a1baae84ad1bcbe39f7d706c5c5
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/25583a1baae84ad1bcbe39f7d706c5c5/output/log
[joint_bone] epoch   0  train_acc=0.2997  val_top1=0.4558  val_mca=0.3966
[joint_bone] epoch  10  train_acc=0.7701  val_top1=0.6953  val_mca=0.6351
[joint_bone] epoch  20  train_acc=0.8979  val_top1=0.7132  val_mca=0.6367
[joint_bone] epoch  30  train_acc=0.9204  val_top1=0.7196  val_mca=0.6334
[joint_bone] epoch  40  train_acc=0.9250  val_top1=0.7119  val_mca=0.6455
[joint_bone] epoch  50  train_acc=0.9230  val_top1=0.7209  val_mca=0.6457
[joint_bone] early stopping at epoch 53, best val_top1=0.7298


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  7.87MB/s]: 


[joint_bone] FINAL: test_top1=0.7020  gap=0.2330  test_mca=0.6196  test_macro_f1=0.6359


### comparison table

In [9]:
LI_ET_AL_BENCHMARK = {"stream": "Li et al. benchmark (single model)", "test_top1": 0.7441,
                      "test_top5": 0.9376, "test_mca": 0.6144, "test_macro_f1": None, "train_test_gap": None}

comparison = pd.DataFrame(list(all_results.values()) + [LI_ET_AL_BENCHMARK]).set_index("stream")
comparison = comparison[["train_acc", "test_top1", "train_test_gap", "test_top5", "test_mca", "test_macro_f1"]]
comparison.to_csv(os.path.join(OUT_ROOT, "stream_comparison.csv"))
comparison


,train_acc,test_top1,train_test_gap,test_top5,test_mca,test_macro_f1
stream,,,,,,
joint,0.978403,0.700767,0.277636,0.932225,0.593775,0.615867
bone,0.874580,0.682864,0.191716,0.927110,0.566022,0.579301
joint_bone,0.935050,0.702046,0.233004,0.939898,0.619550,0.635879
Li et al. benchmark (single model),NaN,0.744100,NaN,0.937600,0.614400,NaN


---

Every run's confusion matrix and per-class table are now on that run's ClearML task page, not just the training curve. If `bone` or `joint_bone` genuinely beats `joint` here, that mirrors mmaction2's own published NTU60 finding and is a real, defensible single-model improvement - not a trick, just richer input to the same architecture.